In [1]:
import pandas as pd
import numpy as np

# ==========================
# CONFIG
# ==========================

GMV_FILE = "./raw_data/SM_HN_HCM_REV_filtered_2026_5.csv"
LEADS_FILE = "./raw_data/PalFish - Chatpage 2026 - Trực page T01-T05_2026.csv"

GMV_PHONE_COL = "Phone"
GMV_UID_COL = "UID"

LEADS_PHONE_COL = "Số điện thoại"
LEADS_UID_COL = "UID"


# ==========================
# CLEAN UID
# ==========================

def clean_uid(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )

    s = s.replace("", np.nan)

    return s


# ==========================
# CLEAN PHONE
# ==========================

def clean_phone(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\D", "", regex=True)
    )

    # quá ngắn hoặc quá dài -> invalid
    s = s.where(s.str.len().between(9, 13), np.nan)

    return s


# ==========================
# LOAD
# ==========================

df_gmv = pd.read_csv(GMV_FILE)
df_leads = pd.read_csv(LEADS_FILE)


# ==========================
# CLEAN
# ==========================

df_gmv["UID_clean"] = clean_uid(df_gmv[GMV_UID_COL])
df_gmv["Phone_clean"] = clean_phone(df_gmv[GMV_PHONE_COL])

df_leads["UID_clean"] = clean_uid(df_leads[LEADS_UID_COL])
df_leads["Phone_clean"] = clean_phone(df_leads[LEADS_PHONE_COL])


# ==========================
# EXPORT
# ==========================

df_gmv.to_csv(
    "./cleaned_data/gmv_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_leads.to_csv(
    "./cleaned_data/leads_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done!")

Done!


In [ ]:
df_gmv = pd.read_csv("./cleaned_data/gmv_clean.csv", encoding='utf-8-sig')
df_leads = pd.read_csv("./cleaned_data/leads_clean.csv", encoding='utf-8-sig')

In [ ]:
df_gmv.head()

In [4]:
print(df_gmv["TEAM"].unique())

<StringArray>
['HCM', 'In-house', 'In-house 2', 'Linh Dam Store']
Length: 4, dtype: str


In [ ]:
unique_prefix = (
    df_leads["TVTS"]
    .dropna()
    .str.split("-", n=1)
    .str[0]
    .str.strip()
    .unique()
)

print(unique_prefix)

['HN' 'HN2' 'HCM' 'OFF' 'HuyenKTT' 'LyBP' 'AnhVP' 'kho' 'Phuong' 'tr'
 'IND' 'TamNTT' 'TrangDT' 'LinhNTT' 'ThinhND']


In [15]:
# Drop NaN values to avoid errors during string operations
tvts_series = df_leads["TVTS"].dropna()

# Strip spaces and filter the Series for values that start with 'OFF'
off_values = tvts_series[tvts_series.str.strip().str.startswith("OFF")]

# If you want to see all occurrences
print(len(off_values))

# If you only want the unique 'OFF - something' string values
unique_off_values = off_values.unique()
print(unique_off_values)

50
<StringArray>
['OFF - Julia']
Length: 1, dtype: str


In [ ]:
import pandas as pd
import numpy as np

# 1. Define the exact mapping for the 4 TEAM cases
team_mapping = {
    "In-house": "HN",
    "In-house 2": "HN2",
    "HCM": "HCM",
    "Linh Dam Store": "OFF"
}

# 2. Function to automatically format Vietnamese names
def format_vietnamese_name(name):
    # Handle empty or NaN values
    if pd.isna(name):
        return ""
    
    # Strip leading/trailing spaces
    name = str(name).strip()
    parts = name.split()
    
    # If the name has spaces (e.g., "Cao Thi Lua"), convert to "LuaCT"
    if len(parts) > 1:
        # Take the last word as the main name
        first_name = parts[-1]
        # Get the first letter of all preceding words and uppercase them
        initials = "".join([p[0].upper() for p in parts[:-1]])
        return f"{first_name}{initials}"
    
    # If it's already a single string without spaces (e.g., "LinhNT"), return as is
    return name

# 3. Create the new column for abbreviated Sales names ONLY
df_gmv["Sales_Abbr"] = df_gmv["Sales"].apply(format_vietnamese_name)

# 4. Apply the team mapping to get the corresponding prefixes
prefixes = df_gmv["TEAM"].map(team_mapping)

# 5. Define conditions and choices for the 'Sales_infor' column
conditions = [
    # Condition 1: If TEAM is Linh Dam Store, hardcode the value to 'OFF - Julia'
    df_gmv["TEAM"] == "Linh Dam Store",
    
    # Condition 2: For other mapped teams, combine prefix and the abbreviated name
    prefixes.notna()
]

choices = [
    # Output for Condition 1
    "OFF - Julia",
    
    # Output for Condition 2
    prefixes + " - " + df_gmv["Sales_Abbr"]
]

# 6. Apply np.select to create the 'Sales_infor' column based on the conditions
# Teams not in the mapping will get NaN
df_gmv["Sales_infor"] = np.select(conditions, choices, default=np.nan)

# Print a sample to verify both new columns
print(df_gmv[["TEAM", "Sales", "Sales_Abbr", "Sales_infor"]])

# Export to CSV
df_gmv.to_csv('./cleaned_data/gmv_clean_sales_infor.csv', encoding='utf-8-sig', index=False)

           TEAM               Sales Sales_Abbr     Sales_infor
0           HCM    Nguyen Minh Phat     PhatNM    HCM - PhatNM
1           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
2           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
3           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
4           HCM  Lai Ngoc Thuy Linh    LinhLNT   HCM - LinhLNT
..          ...                 ...        ...             ...
458    In-house  Luu Thi Hoang Ngan    NganLTH    HN - NganLTH
459    In-house  Luu Thi Hoang Ngan    NganLTH    HN - NganLTH
460    In-house   Nguyen Kieu Trang    TrangNK    HN - TrangNK
461  In-house 2   Vu Ho Thanh Huong   HuongVHT  HN2 - HuongVHT
462  In-house 2   Vu Ho Thanh Huong   HuongVHT  HN2 - HuongVHT

[463 rows x 4 columns]


In [17]:
# Group the dataframe by 'TEAM' and extract the first 3 rows of each group
# We only select the relevant columns for a cleaner output
sample_check = df_gmv[["TEAM", "Sales", "Sales_Abbr", "Sales_infor"]].groupby("TEAM").head(3)

# Display the result to verify the applied logic
print(sample_check)

               TEAM               Sales Sales_Abbr    Sales_infor
0               HCM    Nguyen Minh Phat     PhatNM   HCM - PhatNM
1               HCM  Lai Ngoc Thuy Linh    LinhLNT  HCM - LinhLNT
2               HCM  Lai Ngoc Thuy Linh    LinhLNT  HCM - LinhLNT
19         In-house         Cao Thi Lua      LuaCT     HN - LuaCT
20         In-house        Le Thi Tuyet    TuyetLT   HN - TuyetLT
21         In-house   Pham Thi Tam Thuy    ThuyPTT   HN - ThuyPTT
34       In-house 2         Le Thi Tram     TramLT   HN2 - TramLT
35       In-house 2           Vu Cam Ly       LyVC     HN2 - LyVC
48       In-house 2           Vu Cam Ly       LyVC     HN2 - LyVC
56   Linh Dam Store   Ngo Thi Thuy Linh    LinhNTT    OFF - Julia
123  Linh Dam Store      Nguyen Thi Lan      LanNT    OFF - Julia
127  Linh Dam Store      Nguyen Thi Lan      LanNT    OFF - Julia


In [21]:
df_gmv = pd.read_csv('./cleaned_data/gmv_clean_sales_infor.csv', encoding='utf-8-sig')
df_leads = pd.read_csv("./cleaned_data/leads_clean.csv", encoding='utf-8-sig')

In [3]:
print(df_gmv[["UID_clean", "Phone_clean"]].head(10))
print(df_leads[["UID_clean", "Phone_clean"]].head(10))

    UID_clean  Phone_clean
0  3311001921  84908224672
1  3179205818  84909517679
2  3311304274  84964678701
3  3310902627  84389970625
4  3309271098  84938593961
5  3311605431  84939899045
6  3310167280  84978827804
7  3311379165  84948063983
8  3311850607  84982135774
9  3311951330  84938222653
    UID_clean    Phone_clean
0                           
1  3310947243  84919249237.0
2  3310952712  84965468525.0
3  3293618952  84985951781.0
4  3310947249  84972856556.0
5  3310952706  84907278168.0
6  3310947256  84388880693.0
7  3310947803  84968455579.0
8  3310947816  84914905977.0
9  3310947809  84387666764.0


In [5]:
import pandas as pd

# Đọc dữ liệu
df_gmv = pd.read_csv("./cleaned_data/gmv_clean.csv", encoding="utf-8-sig")
df_leads = pd.read_csv("./cleaned_data/leads_clean.csv", encoding="utf-8-sig")

# ==========================
# Chuẩn hóa dữ liệu
# ==========================
for df in [df_gmv, df_leads]:
    for col in ["UID_clean", "Phone_clean"]:
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
            .str.replace(".0", "", regex=False)   # bỏ .0
            .str.strip()                          # bỏ khoảng trắng
        )

# (Tùy chọn) Loại bỏ dòng mà cả UID và Phone đều rỗng
df_gmv = df_gmv[
    (df_gmv["UID_clean"] != "") |
    (df_gmv["Phone_clean"] != "")
]

df_leads = df_leads[
    (df_leads["UID_clean"] != "") |
    (df_leads["Phone_clean"] != "")
]

# ==========================
# Tạo set (UID, Phone)
# ==========================
gmv_pairs = set(zip(df_gmv["UID_clean"], df_gmv["Phone_clean"]))
leads_pairs = set(zip(df_leads["UID_clean"], df_leads["Phone_clean"]))

# Giao nhau
common_pairs = gmv_pairs & leads_pairs

# ==========================
# Kết quả
# ==========================
print(f"Số cặp (UID, Phone) trong GMV   : {len(gmv_pairs)}")
print(f"Số cặp (UID, Phone) trong Leads : {len(leads_pairs)}")
print(f"Số cặp giao nhau                : {len(common_pairs)}")

print("\n20 cặp giao nhau đầu tiên:")
for pair in list(common_pairs)[:20]:
    print(pair)

# ==========================
# Debug thêm
# ==========================
uid_common = set(df_gmv["UID_clean"]) & set(df_leads["UID_clean"])
phone_common = set(df_gmv["Phone_clean"]) & set(df_leads["Phone_clean"])

print("\n==============================")
print(f"UID common   : {len(uid_common)}")
print(f"Phone common : {len(phone_common)}")

Số cặp (UID, Phone) trong GMV   : 457
Số cặp (UID, Phone) trong Leads : 15828
Số cặp giao nhau                : 260

20 cặp giao nhau đầu tiên:
('3311100120', '821056952019')
('3311600042', '4917630176062')
('3311268494', '84344941261')
('3311911531', '84966652526')
('3311232054', '84784949889')
('3311109861', '84981763246')
('3311952734', '4917657886868')
('3311447351', '84342856796')
('3310292291', '84837481789')
('3311379165', '84948063983')
('3307729617', '84372211018')
('3312156296', '84986053049')
('3312172633', '84917545333')
('3310924924', '84948976742')
('3310676174', '84349585875')
('3311304271', '84369846754')
('3298131980', '84936454433')
('3310712174', '84976202737')
('3311357753', '818099900031')
('3309886077', '84938345968')

UID common   : 260
Phone common : 279


In [7]:
import pandas as pd

# ==========================
# Read data
# ==========================
df_gmv = pd.read_csv("./cleaned_data/gmv_clean_sales_infor.csv", encoding="utf-8-sig")
df_leads = pd.read_csv("./cleaned_data/leads_clean.csv", encoding="utf-8-sig")

# ==========================
# Normalize
# ==========================

# GMV
for col in ["UID_clean", "Phone_clean", "Sales_infor"]:
    df_gmv[col] = (
        df_gmv[col]
        .fillna("")
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

# Leads
for col in ["UID_clean", "Phone_clean", "TVTS"]:
    df_leads[col] = (
        df_leads[col]
        .fillna("")
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

# Loại bỏ dòng không có cả UID và Phone
df_gmv = df_gmv[
    (df_gmv["UID_clean"] != "") |
    (df_gmv["Phone_clean"] != "")
].copy()

df_leads = df_leads[
    (df_leads["UID_clean"] != "") |
    (df_leads["Phone_clean"] != "")
].copy()

# ==========================
# Build result
# ==========================
matched_rows = []
unmatched_rows = []

matched_count = 0

# Các cột của leads sẽ được nối thêm vào GMV
lead_columns = list(df_leads.columns)

for idx, gmv_row in df_gmv.iterrows():

    uid = gmv_row["UID_clean"]
    phone = gmv_row["Phone_clean"]
    sales = gmv_row["Sales_infor"]

    # --------------------------
    # Rule 1: UID + Phone
    # --------------------------
    candidates = df_leads[
        (df_leads["UID_clean"] == uid)
        &
        (df_leads["Phone_clean"] == phone)
    ]

    reason = ""

    if len(candidates) == 0:
        # Không match
        new_row = gmv_row.to_dict()

        for col in lead_columns:
            new_row[f"Lead_{col}"] = pd.NA

        new_row["Match_Status"] = "Unmatched"
        new_row["Match_Reason"] = "No UID+Phone"

        unmatched_rows.append(new_row)

    else:

        if len(candidates) == 1:

            chosen = candidates.iloc[0]
            reason = "Unique UID+Phone"

        else:

            # Ưu tiên TVTS == Sales_infor
            same_sales = candidates[
                candidates["TVTS"] == sales
            ]

            if len(same_sales) > 0:

                chosen = same_sales.iloc[0]
                reason = "Matched by Sales_infor"

            else:

                chosen = candidates.iloc[0]
                reason = "First candidate"

        new_row = gmv_row.to_dict()

        for col in lead_columns:
            new_row[f"Lead_{col}"] = chosen[col]

        new_row["Match_Status"] = "Matched"
        new_row["Match_Reason"] = reason

        matched_rows.append(new_row)
        matched_count += 1

# ==========================
# Export
# ==========================

matched_df = pd.DataFrame(matched_rows)
unmatched_df = pd.DataFrame(unmatched_rows)

final_df = pd.concat(
    [matched_df, unmatched_df],
    ignore_index=True
)

final_df.to_csv(
    "./output/final_join.csv",
    index=False,
    encoding="utf-8-sig"
)

matched_df.to_csv(
    "./output/matched.csv",
    index=False,
    encoding="utf-8-sig"
)

unmatched_df.to_csv(
    "./output/unmatched.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================
# Statistics
# ==========================
print("=" * 50)
print(f"GMV rows      : {len(df_gmv):,}")
print(f"Matched       : {matched_count:,}")
print(f"Unmatched     : {len(unmatched_df):,}")
print(f"Match rate    : {matched_count/len(df_gmv):.2%}")
print("=" * 50)

print("\nReason:")
print(final_df["Match_Reason"].value_counts())

GMV rows      : 463
Matched       : 262
Unmatched     : 201
Match rate    : 56.59%

Reason:
Match_Reason
Unique UID+Phone          254
No UID+Phone              201
Matched by Sales_infor      5
First candidate             3
Name: count, dtype: int64
